In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [8]:
import asyncio
import os
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def extract_compound_data():
    # File paths
    input_file = 'compound_links.txt'
    output_file = 'compound_data_descriptions.txt'

    # Check if input file exists
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    # Read all links from the file
    with open(input_file, 'r', encoding='utf-8') as f:
        links = [line.strip() for line in f if line.strip().startswith('http')]

    print(f"Total links found: {len(links)}. Starting full extraction...")

    async with async_playwright() as p:
        # Launch browser with robust settings for server/notebook environments
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"])
        context = await browser.new_context()
        page = await context.new_page()

        # Initialize output file (overwrite with header)
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("NAWY COMPOUND DATA EXTRACTION - FULL COLLECTION\n")

        for index, url in enumerate(links, 1):
            compound_slug = url.split('/')[-1]
            print(f"[{index}/{len(links)}] Processing: {compound_slug}...")

            try:
                # Navigate to the compound page
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")

                # Small delay to let dynamic content load
                await page.wait_for_timeout(2000)

                # Handle the "See More" button if it exists to expand description
                try:
                    show_more_btn = page.locator('button[data-test="show-more-less"]')
                    if await show_more_btn.is_visible(timeout=3000):
                        btn_text = await show_more_btn.text_content()
                        if btn_text and "More" in btn_text:
                            await show_more_btn.click()
                            await page.wait_for_timeout(1000)
                except:
                    pass

                # Extract HTML and parse
                html_content = await page.content()
                soup = BeautifulSoup(html_content, 'html.parser')

                # Target the side container
                details_container = soup.select_one('.details-side')

                extracted_text = ""

                if details_container:
                    # Look for the #head div (the "About" section)
                    head_div = details_container.find('div', id='head')
                    if head_div:
                        # Extract the title (e.g., "About Mountain View October Park")
                        title_el = head_div.find(['h1', 'h2', 'h3'], class_='entity-title')
                        title_text = title_el.get_text(strip=True) if title_el else f"About {compound_slug.replace('-', ' ').title()}"

                        # Build formatted header for the compound name with ===
                        header_line = "=" * len(title_text)
                        formatted_title = f"{header_line}\n{title_text}\n{header_line}\n"

                        # Get the rest of the text content within the head div
                        body_text = head_div.get_text(separator='\n', strip=True)
                        # Replace the first instance of the title with our formatted version
                        if title_text in body_text:
                            body_text = body_text.replace(title_text, formatted_title, 1)
                        else:
                            body_text = formatted_title + body_text

                        extracted_text += "--- DESCRIPTION ---\n"
                        extracted_text += body_text + "\n\n"

                # Also check for FAQs outside the details-side if it exists
                faqs_div = soup.find('div', {'data-test': 'faqs'})
                if faqs_div:
                    extracted_text += "--- FAQs ---\n"
                    extracted_text += faqs_div.get_text(separator='\n', strip=True)

                if extracted_text.strip():
                    with open(output_file, 'a', encoding='utf-8') as f:
                        # Link separators using ===
                        f.write(f"\n\n{'='*60}\nURL: {url}\n{'='*60}\n\n")
                        f.write(extracted_text)
                    print(f"  -> SUCCESS: Saved.")
                else:
                    print(f"  -> WARNING: No specific content found.")

            except Exception as e:
                print(f"  -> ERROR: {e}")

        # Final cleanup
        try:
            await browser.close()
        except:
            pass

        print(f"\nExtraction complete! Results are in {output_file}")

# Execution for Jupyter/Colab environment
await extract_compound_data()

Total links found: 1708. Starting full extraction...
[1/1708] Processing: 10-mountain-view-october-park...
  -> SUCCESS: Saved.
[2/1708] Processing: 100-brix...
  -> SUCCESS: Saved.
[3/1708] Processing: 1000-obsidier...
  -> SUCCESS: Saved.
[4/1708] Processing: 1001-il-latini-city-edge...
  -> SUCCESS: Saved.
[5/1708] Processing: 1002-beach-chalets-seashore...
  -> SUCCESS: Saved.
[6/1708] Processing: 1003-ivoire-west...
  -> SUCCESS: Saved.
[7/1708] Processing: 1005-dijar...
  -> SUCCESS: Saved.
[8/1708] Processing: 1006-al-alamein-residences...
  -> WARNING: No specific content found.
[9/1708] Processing: 1007-marina-2...
  -> SUCCESS: Saved.
[10/1708] Processing: 1008-lake-view-residence-2...
  -> SUCCESS: Saved.
[11/1708] Processing: 1009-ira...
  -> SUCCESS: Saved.
[12/1708] Processing: 1010-vigor...
  -> SUCCESS: Saved.
[13/1708] Processing: 1011-regent's-square...
  -> SUCCESS: Saved.
[14/1708] Processing: 1012-mamsha-views...
  -> SUCCESS: Saved.
[15/1708] Processing: 1013-mams